# Config

In [2]:
#pip install transformers datasets torch scikit-learn accelerate

In [40]:
#pip install -U transformers accelerate

# Sentiment analysis

In [41]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score



In [42]:
# =========================
# 1. Chargement des données
# =========================
df = pd.read_csv("../res/train_verbatims_for_sentiment.csv")

label2id = {"NEGATIF": 0, "NEUTRE": 1, "POSITIF": 2}
id2label = {v: k for k, v in label2id.items()}

df.rename(columns={"verbatim":"text","sentiment":"label"},inplace=True)

df["text"] = df["text"].astype(str)

df["label"] = df["label"].map(label2id)

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)


In [43]:

# =========================
# 2. Tokenizer
# =========================
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# =========================
# 3. Modèle
# =========================
model = CamembertForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

# =========================
# 4. Métriques
# =========================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }


Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
# =========================
# 5. Entraînement
# =========================
training_args = TrainingArguments(
    output_dir="./sentiment_camembert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\idris\AppData\Local\Temp\ipykernel_16952\3071561396.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.000500,0.373343,0.948810,0.949163
2,0.011500,0.395598,0.954762,0.955008
3,0.014900,0.394371,0.954762,0.955006
4,0.019000,0.332946,0.959524,0.959788
5,0.026100,0.413729,0.946429,0.946660
6,0.001700,0.375757,0.950000,0.950386
7,0.002600,0.360838,0.958333,0.958611
8,0.004900,0.368810,0.954762,0.955052
9,0.000200,0.374512,0.954762,0.955052
10,0.003200,0.375650,0.957143,0.957421


TrainOutput(global_step=2100, training_loss=0.010956576085605083, metrics={'train_runtime': 312.5672, 'train_samples_per_second': 107.497, 'train_steps_per_second': 6.719, 'total_flos': 2210152708915200.0, 'train_loss': 0.010956576085605083, 'epoch': 10.0})

In [51]:
# =========================
# 6. Sauvegarde
# =========================
trainer.save_model("models/camembert_sentiment_fr")
tokenizer.save_pretrained("models/camembert_sentiment_fr")

('models/camembert_sentiment_fr\\tokenizer_config.json',
 'models/camembert_sentiment_fr\\special_tokens_map.json',
 'models/camembert_sentiment_fr\\sentencepiece.bpe.model',
 'models/camembert_sentiment_fr\\added_tokens.json')

In [52]:
# Load model and tokenizer
model = CamembertForSequenceClassification.from_pretrained("models/camembert_sentiment_fr")
tokenizer = CamembertTokenizer.from_pretrained("models/camembert_sentiment_fr")

def predict_sentiment(texts):
    """
    Predict sentiment for one or multiple texts.
    
    Args:
        texts (str or list): Single text or list of texts to classify
        
    Returns:
        str or list: Sentiment label(s) (NEGATIF, NEUTRE, or POSITIF)
    """
    # Handle single text input
    if isinstance(texts, str):
        texts = [texts]
        single_input = True
    else:
        single_input = False
    
    # Tokenize
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    
    # Get predictions
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = np.argmax(logits.detach().numpy(), axis=1)
    
    # Convert to sentiment labels
    sentiments = [id2label[pred] for pred in predictions]
    
    # Return single result if single input
    return sentiments[0] if single_input else sentiments

In [53]:
# Test the function
texts = [
    "Les notes d'examens doivent être communiquées beaucoup plus rapidement et il faut faire en sorte que chaque étudiant puisse réaliser un examen dans les meilleures conditions de manière équitables (par exemple lors de compréhensions orales en amphithéâtre, ceux de la rangée du fond seront très pénalisés, faute d'enceintes fonctionnelles et jouant peu fort).",
    "Améliorer la communication avec le Career Center, 2000 entreprises partenaires mais tout de même un fort nombre de personnes sans alternances ou stages. En tant qu'étudiant, les entreprises sont là pour faire de la figuration et simplement embellir l'image de l'école",
    "Je suis très déçu par l'organisation",
    "Je suis en train de vivre les plus belles années de ma vie à l'EFREI.",
    "RAS",
    "jsp",
    ".",
    "/////",
    "je suis jolie",
    "je n'ai pas d'avis",
    "C'est correct",
    "arretez de me harceler avec des enquêtes de satisfaction",
    "je n'ai pas d'avis",
    "Arretez de vous foutre de nous",
    "Avant d’organiser qqch demandez si on est d’accord, la qualité des intervenant car ils viennent à l’efrei mais l’efrei n’a aucune idée des cours qui sont donnés ",
    "Plus de vrai cours et moins de intervenants incompétents.  Les cours de com sont dominants et nous distrait de nos disciplines principales ( 50% des heures de cours sont de la com pour le S8 ) . Plus d'enseignants chercheurs experts de leurs domaines et moins de  cours vides et sans intérêts. Plus d'implication dans le monde de la recherche est souhaitable.",
    "je n'ai pas d'avis",
    "J'ai beaucoup apprécié les matières d'ouverture (Communication & SS) de la dernière année, je pense qu'avoir l'occasion d'étudier plus de deux de ces matières en commençant en M1 pourrait être un grand plus. Une intervention comme le Business Game pourrait elle être condensée sur une année.",
    "supprimez LXP c'est inutile"
]

for text in texts:
    sentiment = predict_sentiment(text)
    print(f"Texte: {text} => Sentiment: {sentiment}")

Texte: Les notes d'examens doivent être communiquées beaucoup plus rapidement et il faut faire en sorte que chaque étudiant puisse réaliser un examen dans les meilleures conditions de manière équitables (par exemple lors de compréhensions orales en amphithéâtre, ceux de la rangée du fond seront très pénalisés, faute d'enceintes fonctionnelles et jouant peu fort). => Sentiment: NEGATIF
Texte: Améliorer la communication avec le Career Center, 2000 entreprises partenaires mais tout de même un fort nombre de personnes sans alternances ou stages. En tant qu'étudiant, les entreprises sont là pour faire de la figuration et simplement embellir l'image de l'école => Sentiment: NEGATIF
Texte: Je suis très déçu par l'organisation => Sentiment: NEGATIF
Texte: Je suis en train de vivre les plus belles années de ma vie à l'EFREI. => Sentiment: POSITIF
Texte: RAS => Sentiment: NEUTRE
Texte: jsp => Sentiment: NEUTRE
Texte: . => Sentiment: NEUTRE
Texte: ///// => Sentiment: NEUTRE
Texte: je suis jolie =